# 11 — Two-Stage OTA

**Dataset:** `datasets/opamp_twostage/`

Topology:
- Stage 1 (input): 6 NMOS (CM bias + ABBA diff pair) + 4 PMOS (cross-coupled load)
- Stage 2 (gain):  2 PMOS (CS amp) + 2 NMOS (CS bias)

Nodes: `VDD GND DIFFPAIR_BIAS VP VN CS_BIAS VOUT`

In [ ]:
import sys, os
os.environ.setdefault('PDK_ROOT', os.path.expanduser('~/pdks'))
sys.path.insert(0, os.path.abspath('../../src/gelochip'))
import gelochip.gl as gl
gl.reload()  # pick up latest code without restarting kernel


In [ ]:
vp          = gl.Net('vp')
vn          = gl.Net('vn')
dp_bias     = gl.Net('dp_bias')   # diff pair bias (DIFFPAIR_BIAS)
cs_bias     = gl.Net('cs_bias')   # CS stage bias
vout        = gl.Net('vout')
vtail       = gl.Net('vtail')     # diff pair tail
vdd1        = gl.Net('vdd1')      # left diff drain
vdd2        = gl.Net('vdd2')      # right diff drain (→ Stage 2 input)
v1          = gl.Net('v1')        # diff-to-single internal
vss2        = gl.Net('vss2')      # diff-to-single internal

# ── Stage 1: NMOS diff pair + current mirror bias ────────────────────
m_ref  = gl.nmos(w=6.0, fingers=1, g=dp_bias, d=dp_bias, s=gl.gnd)  # CM ref diode
m_copy = gl.nmos(w=6.0, fingers=1, g=dp_bias, d=vtail,   s=gl.gnd)  # CM copy → tail
m_tl   = gl.nmos(w=6.0, fingers=4, g=vp, d=vdd1, s=vtail)  # A top
m_bl   = gl.nmos(w=6.0, fingers=4, g=vp, d=vdd1, s=vtail)  # A bot
m_tr   = gl.nmos(w=6.0, fingers=4, g=vn, d=vdd2, s=vtail)  # B top
m_br   = gl.nmos(w=6.0, fingers=4, g=vn, d=vdd2, s=vtail)  # B bot

# ── Stage 1: PMOS diff-to-single cross-coupled load ──────────────────
# XTOP1: D=V1,   G=VDD1, S=VDD (VDD=source of PMOS)
# XTOP2: D=VSS2, G=VDD1, S=VDD
# XBOT1: D=VDD1, G=VDD1, S=V1   (diode)
# XBOT2: D=VDD2, G=VDD1, S=VSS2 (converts diff to VDD2=vout_stage1)
mp_t1  = gl.pmos(w=6.0, fingers=6, g=vdd1, d=v1,   s=gl.vdd)
mp_t2  = gl.pmos(w=6.0, fingers=6, g=vdd1, d=vss2, s=gl.vdd)
mp_b1  = gl.pmos(w=6.0, fingers=6, g=vdd1, d=vdd1, s=v1)     # diode-like
mp_b2  = gl.pmos(w=6.0, fingers=6, g=vdd1, d=vdd2, s=vss2)   # converts diff→vdd2

# ── Stage 2: Common-source amp (PMOS) + bias (NMOS) ─────────────────
mp_cs1 = gl.pmos(w=7.0, fingers=10, g=vdd2, d=vout, s=gl.vdd)  # left CS
mp_cs2 = gl.pmos(w=7.0, fingers=10, g=vdd2, d=vout, s=gl.vdd)  # right CS
mn_cs1 = gl.nmos(w=6.0, fingers=8,  g=cs_bias, d=vout, s=gl.gnd)  # left bias
mn_cs2 = gl.nmos(w=6.0, fingers=8,  g=cs_bias, d=vout, s=gl.gnd)  # right bias

chip = gl.build(
    m_ref, m_copy, m_tl, m_bl, m_tr, m_br,
    mp_t1, mp_t2, mp_b1, mp_b2,
    mp_cs1, mp_cs2, mn_cs1, mn_cs2,
    name='opamp_twostage'
)
chip.show()
chip.drc()
chip.sim()